In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df = pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [11]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df.drop("target", axis=1).values
y = df["target"].values

m = len(X)
train_size = int(0.8 * m)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

In [12]:
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [13]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [14]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    cost = 0
    
    for i in range(m):
        z = np.dot(X[i], w) + b
        f_wb = sigmoid(z)
        
        cost += -y[i] * np.log(f_wb) - (1 - y[i]) * np.log(1 - f_wb)
    
    return cost / m

In [15]:
def compute_gradient(X, y, w, b):
    m, n = X.shape
    
    dj_dw = np.zeros(n)
    dj_db = 0
    
    for i in range(m):
        z = np.dot(X[i], w) + b
        f_wb = sigmoid(z)
        err = f_wb - y[i]
        
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        
        dj_db += err
    
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    
    return dj_dw, dj_db

In [16]:
def gradient_descent(X, y, w, b, alpha, iterations):
    cost_history = []
    
    for i in range(iterations):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        
        if i % 100 == 0:
            cost = compute_cost(X, y, w, b)
            cost_history.append(cost)
            print(f"Iteration {i}: Cost = {cost}")
    
    return w, b, cost_history

In [17]:
n = X_train.shape[1]
w = np.zeros(n)
b = 0

In [18]:
alpha = 0.01
iterations = 5000

w, b, cost_history = gradient_descent(X_train, y_train, w, b, alpha, iterations)

Iteration 0: Cost = 0.6900583134308558
Iteration 100: Cost = 0.5129558482040618
Iteration 200: Cost = 0.449227466497396
Iteration 300: Cost = 0.41880718590327926
Iteration 400: Cost = 0.40154915556299975
Iteration 500: Cost = 0.39065547099207054
Iteration 600: Cost = 0.383273653679683
Iteration 700: Cost = 0.37801472717730744
Iteration 800: Cost = 0.3741267890564231
Iteration 900: Cost = 0.3711695110994296
Iteration 1000: Cost = 0.3688689744868863
Iteration 1100: Cost = 0.36704645981914774
Iteration 1200: Cost = 0.36558078590669163
Iteration 1300: Cost = 0.36438714219507734
Iteration 1400: Cost = 0.3634045798000084
Iteration 1500: Cost = 0.36258830266151704
Iteration 1600: Cost = 0.3619047449204168
Iteration 1700: Cost = 0.36132833024891897
Iteration 1800: Cost = 0.3608392814469021
Iteration 1900: Cost = 0.36042210543381203
Iteration 2000: Cost = 0.36006452391257165
Iteration 2100: Cost = 0.35975670489010897
Iteration 2200: Cost = 0.3594907014467353
Iteration 2300: Cost = 0.35926003587

In [19]:
def predict(X, w, b):
    m = X.shape[0]
    predictions = np.zeros(m)
    
    for i in range(m):
        z = np.dot(X[i], w) + b
        f_wb = sigmoid(z)
        
        if f_wb >= 0.5:
            predictions[i] = 1
        else:
            predictions[i] = 0
    
    return predictions

In [20]:
y_pred = predict(X_test, w, b)

In [22]:
accuracy = np.mean(y_pred == y_test)
print("\nFinal Accuracy:", accuracy)


Final Accuracy: 0.8688524590163934


In [23]:
tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

print("\nConfusion Matrix")
print([[tn, fp],
       [fn, tp]])


Confusion Matrix
[[np.int64(24), np.int64(5)], [np.int64(3), np.int64(29)]]


In [25]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Precision: 0.8529411764705882
Recall: 0.90625
F1 Score: 0.8787878787878787


In [26]:
print("\nWeights:")
print(w)

print("\nBias:")
print(b)


Weights:
[-0.22263504 -0.9025582   0.75219509 -0.21025421 -0.22650914  0.00172857
  0.23373524  0.44169792 -0.48428865 -0.66628265  0.34744381 -0.76545181
 -0.49854836]

Bias:
0.1850923785587103


In [27]:
results = pd.DataFrame({
    "True Value": y_test,
    "Predicted Value": y_pred.astype(int)
})

results["Correct?"] = results["True Value"] == results["Predicted Value"]

print(results)

    True Value  Predicted Value  Correct?
0            1                1      True
1            0                0      True
2            0                0      True
3            0                1     False
4            0                0      True
..         ...              ...       ...
56           0                1     False
57           1                1      True
58           1                1      True
59           0                0      True
60           1                1      True

[61 rows x 3 columns]
